# M3 — E05a: multilingual_e5_large_instruct (Kaggle)

**Цель.** Измерить zero-shot direct dense retrieval только для
`multilingual_e5_large_instruct` на frozen M0 validation proxy. Основная метрика — macro
**Recall@50**. Этот запуск изолирован: его результаты не зависят от других
моделей и сохраняются в `/kaggle/working/m3_dense__multilingual_e5_large_instruct`.

Перед запуском включите **Internet** и **GPU accelerator**. Не сохраняйте
эмбеддинги или индексы: сохраняются только метрики, manifest и top-200
validation-кандидаты.

## План ноутбука

1. Установить зависимости и проверить Kaggle GPU.
2. Прочитать Input и инициализировать live ClearML без автоматических
   Jupyter hooks.
3. Восстановить frozen M0 proxy и закодировать только `multilingual_e5_large_instruct`.
4. Измерить Recall@K, сохранить candidates, `metrics.csv` и manifest в
   `/kaggle/working/m3_dense__multilingual_e5_large_instruct`.

## 1. Установка и окружение Kaggle

In [ ]:
# Do not use --upgrade: Kaggle's existing compiled NumPy / scikit-learn
# stack must remain intact. Plain pip install provides any missing
# dependencies (including ClearML's pathlib2) without replacing packages
# that already satisfy the constraints.
!pip install -q "clearml>=1.16" "sentence-transformers>=3.4" "transformers>=4.51"

In [ ]:
from __future__ import annotations

import gc
import importlib.metadata
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import sklearn
SEED = 42
np.random.seed(SEED)
TOP_K = 200
FINAL_K = 50
DESCRIPTION_CHAR_LIMIT = 1_500
QUERY_SCORE_BATCH_SIZE = 128
# Encode one model's texts data-parallel across cuda:0 and cuda:1 when available.
USE_ALL_VISIBLE_GPUS = True
MODEL_NAME = "multilingual_e5_large_instruct"
MODEL_NAMES = (MODEL_NAME,)
OUTPUT_DIR = Path("/kaggle/working/m3_dense__multilingual_e5_large_instruct")
# /kaggle/working is persisted by a saved version; model cache is not an artifact.
HF_CACHE_DIR = Path("/kaggle/temp/hf-cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.pop("CLEARML_OFFLINE_MODE", None)
NOTEBOOK_STARTED = time.perf_counter()

print({
    "output_dir": str(OUTPUT_DIR),
    "models": list(MODEL_NAMES),
    "scikit_learn": sklearn.__version__,
    "pyarrow": pyarrow.__version__,
})

## 2. Input dataset, ClearML Secrets и общий retrieval-код

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_required_secret(name: str) -> str:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Add Kaggle Secret {name!r} before running this notebook.") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty.")
    return value


def get_optional_secret(name: str) -> None:
    try:
        value = secrets.get_secret(name)
    except Exception:
        return
    if value:
        os.environ[name] = value


os.environ["CLEARML_API_ACCESS_KEY"] = get_required_secret("CLEARML_API_ACCESS_KEY")
os.environ["CLEARML_API_SECRET_KEY"] = get_required_secret("CLEARML_API_SECRET_KEY")
for optional_name in ("CLEARML_API_HOST", "CLEARML_WEB_HOST", "CLEARML_FILES_HOST"):
    get_optional_secret(optional_name)

input_root = Path("/kaggle/input")

def find_input_file(filename: str) -> Path:
    matches = sorted(input_root.glob(f"**/{filename}"))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one attached Kaggle Input named {filename!r}, found: {[str(path) for path in matches]}"
        )
    return matches[0]


TRAIN_PATH = find_input_file("train.parquet")
BENCHMARK_QUERIES_PATH = find_input_file("benchmark_queries.parquet")
BENCHMARK_ITEMS_PATH = find_input_file("benchmark_items.parquet")
print({
    "train": str(TRAIN_PATH),
    "benchmark_queries": str(BENCHMARK_QUERIES_PATH),
    "benchmark_items": str(BENCHMARK_ITEMS_PATH),
    "clearml_credentials_present": True,
})

In [ ]:
from clearml import Task

clearml_task = Task.init(
    project_name="avito-retrieval",
    task_name="E05a__multilingual_e5_large_instruct__kaggle__s42",
    reuse_last_task_id=False,
    # The prior combined run stalled in ClearML's Jupyter hooks.
    # Metrics are reported manually below, so disable only auto-hooks.
    auto_connect_arg_parser=False,
    auto_connect_frameworks={"detect_repository": False},
    auto_resource_monitoring=False,
    auto_connect_streams=False,
)
clearml_task.connect(
    {
        "stage": "M3_E05a_zero_shot_direct_dense",
        "environment": "kaggle",
        "validation_protocol": "benchmark_aligned_proxy_v1",
        "seed": SEED,
        "top_k": TOP_K,
        "final_k": FINAL_K,
        "description_char_limit": DESCRIPTION_CHAR_LIMIT,
        "query_score_batch_size": QUERY_SCORE_BATCH_SIZE,
        "use_all_visible_gpus": USE_ALL_VISIBLE_GPUS,
        "category_rule": "item_category_id == search_category; full-corpus fallback when absent",
        "model": MODEL_NAME,
        "models": list(MODEL_NAMES),
        "shared_source_sha256": 'd7d8dcca6d65f25d909730f8f1fcef1245fe9253d25a80e53024a666e1a089f7',
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_task_id": clearml_task.id, "offline_mode": False, "clearml_hooks": "manual"})

## 3. Frozen M0 proxy и набор bi-encoders

In [ ]:
# Shared M3 utilities snapshot from the repository. SHA256: d7d8dcca6d65f25d909730f8f1fcef1245fe9253d25a80e53024a666e1a089f7
# Make the snapshot importable without cloning the Git repository.
import sys
import types

shared_module = types.ModuleType('dense_retrieval')
exec('"""Shared, exact dense-retrieval utilities for M3 notebooks.\n\nThe module deliberately has no ClearML dependency.  Notebooks own experiment\ntracking; this module owns the frozen proxy, text construction and exact\ncategory-partitioned search so local and Kaggle runs cannot silently diverge.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import GroupShuffleSplit\n\n\nSEARCH_COLUMNS = [\n    "search_query",\n    "search_location_id",\n    "search_is_delivery_search",\n    "search_infm_params_text",\n    "search_category",\n]\nTRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]\nBENCHMARK_QUERY_COLUMNS = ["query_id", *SEARCH_COLUMNS]\nITEM_COLUMNS = [\n    "item_id",\n    "item_title_raw",\n    "item_description_raw",\n    "item_infm_params_text",\n    "item_category_id",\n]\nMETRIC_KS = (1, 5, 10, 20, 50, 200)\n\n# The variants differ by training data, architecture and prompting strategy.\n# All are open-weight, local models; public leaderboards only define a shortlist.\nMODEL_SPECS: dict[str, dict[str, Any]] = {\n    "multilingual_e5_large_instruct": {\n        "model_id": "intfloat/multilingual-e5-large-instruct",\n        "query_mode": "e5_instruction",\n        "query_instruction": "Given a Russian service-search request, retrieve relevant service listings.",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "bge_m3": {\n        "model_id": "BAAI/bge-m3",\n        "query_mode": "plain",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "user_bge_m3": {\n        "model_id": "deepvk/USER-bge-m3",\n        "query_mode": "plain",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "ru_en_rosberta": {\n        "model_id": "ai-forever/ru-en-RoSBERTa",\n        "query_mode": "prefix",\n        "query_prefix": "search_query: ",\n        "document_prefix": "search_document: ",\n        "max_length": 256,\n        "batch_size": 48,\n    },\n    "qwen3_embedding_0_6b": {\n        "model_id": "Qwen/Qwen3-Embedding-0.6B",\n        "query_mode": "sentence_transformers_prompt",\n        "query_prompt_name": "query",\n        "max_length": 256,\n        "batch_size": 16,\n    },\n}\n\n\ndef canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:\n    """Canonicalize only fields used to identify a logical query group."""\n    result = frame[SEARCH_COLUMNS].copy()\n    for column in ("search_query", "search_infm_params_text"):\n        result[column] = (\n            result[column]\n            .astype("string")\n            .fillna("<NA>")\n            .str.lower()\n            .str.strip()\n            .str.replace(r"\\s+", " ", regex=True)\n        )\n    for column in ("search_location_id", "search_is_delivery_search", "search_category"):\n        result[column] = result[column].astype("string").fillna("<NA>")\n    return result\n\n\ndef load_frozen_proxy(\n    train_path: Path,\n    benchmark_queries_path: Path,\n    benchmark_items_path: Path,\n    *,\n    seed: int,\n) -> dict[str, Any]:\n    """Recreate the M0 benchmark-aligned, group-disjoint validation proxy."""\n    train_pairs = pd.read_parquet(train_path, columns=TRAIN_COLUMNS)\n    benchmark_queries = pd.read_parquet(benchmark_queries_path, columns=BENCHMARK_QUERY_COLUMNS)\n    candidate_items = pd.read_parquet(benchmark_items_path, columns=ITEM_COLUMNS).reset_index(drop=True)\n\n    all_contexts = pd.concat(\n        [canonical_query_frame(train_pairs), canonical_query_frame(benchmark_queries)],\n        ignore_index=True,\n    )\n    group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)\n    train_pairs["query_group"] = group_ids[: len(train_pairs)]\n\n    candidate_item_ids = candidate_items["item_id"].astype(str).to_numpy()\n    candidate_item_id_set = set(candidate_item_ids)\n    proxy_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(candidate_item_id_set)].copy()\n\n    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)\n    train_idx, valid_idx = next(splitter.split(proxy_pairs, groups=proxy_pairs["query_group"]))\n    proxy_train_pairs = proxy_pairs.iloc[train_idx].copy()\n    proxy_valid_pairs = proxy_pairs.iloc[valid_idx].copy()\n    assert set(proxy_train_pairs["query_group"]).isdisjoint(set(proxy_valid_pairs["query_group"]))\n\n    validation_queries = (\n        proxy_valid_pairs.sort_values("query_group")\n        .drop_duplicates("query_group")\n        .loc[:, ["query_group", *SEARCH_COLUMNS]]\n        .reset_index(drop=True)\n    )\n    gold_by_group = (\n        proxy_valid_pairs.groupby("query_group", sort=False)["item_id"]\n        .agg(lambda values: frozenset(values.astype(str)))\n        .to_dict()\n    )\n    gold_sets = [gold_by_group[group] for group in validation_queries["query_group"]]\n\n    category_to_indices = {\n        str(category): group.index.to_numpy(dtype=np.int64)\n        for category, group in candidate_items.groupby("item_category_id", sort=False)\n    }\n    all_indices = np.arange(len(candidate_items), dtype=np.int64)\n    allowed_indices_by_query = [\n        category_to_indices.get(str(category), all_indices)\n        for category in validation_queries["search_category"]\n    ]\n    category_oracle = float(\n        np.mean(\n            [\n                len(gold & set(candidate_item_ids[allowed])) / len(gold)\n                for gold, allowed in zip(gold_sets, allowed_indices_by_query, strict=True)\n            ]\n        )\n    )\n    assert category_oracle == 1.0, "The M0 category partition removed a validation positive."\n\n    return {\n        "train_pairs": train_pairs,\n        "proxy_train_pairs": proxy_train_pairs,\n        "proxy_valid_pairs": proxy_valid_pairs,\n        "validation_queries": validation_queries,\n        "gold_sets": gold_sets,\n        "candidate_items": candidate_items,\n        "candidate_item_ids": candidate_item_ids,\n        "category_to_indices": category_to_indices,\n        "all_indices": all_indices,\n        "category_oracle": category_oracle,\n        "benchmark_queries": benchmark_queries,\n    }\n\n\ndef compose_query_text(frame: pd.DataFrame) -> list[str]:\n    """Keep the request primary; filters are useful but must not replace it."""\n    query = frame["search_query"].fillna("").astype(str).str.strip()\n    filters = frame["search_infm_params_text"].fillna("").astype(str).str.strip()\n    return [\n        value if not params else f"{value}\\nФильтры поиска: {params}"\n        for value, params in zip(query, filters, strict=True)\n    ]\n\n\ndef compose_item_text(frame: pd.DataFrame, *, description_char_limit: int = 1_500) -> list[str]:\n    """Use only user-visible item text; cap descriptions before tokenization."""\n    title = frame["item_title_raw"].fillna("").astype(str).str.strip()\n    params = frame["item_infm_params_text"].fillna("").astype(str).str.strip()\n    description = frame["item_description_raw"].fillna("").astype(str).str.slice(stop=description_char_limit).str.strip()\n    texts: list[str] = []\n    for title_value, params_value, description_value in zip(title, params, description, strict=True):\n        fields = [f"Название услуги: {title_value}"]\n        if params_value:\n            fields.append(f"Параметры: {params_value}")\n        if description_value:\n            fields.append(f"Описание: {description_value}")\n        texts.append("\\n".join(fields))\n    return texts\n\n\ndef prepare_model_texts(texts: list[str], spec: dict[str, Any], *, is_query: bool) -> tuple[list[str], dict[str, Any]]:\n    """Apply only the prompt format declared by each model\'s model card."""\n    if not is_query:\n        prefix = str(spec.get("document_prefix", ""))\n        return [prefix + text for text in texts], {}\n\n    mode = spec["query_mode"]\n    if mode == "plain":\n        return texts, {}\n    if mode == "e5_instruction":\n        instruction = str(spec["query_instruction"])\n        return [f"Instruct: {instruction}\\nQuery: {text}" for text in texts], {}\n    if mode == "prefix":\n        return [str(spec["query_prefix"]) + text for text in texts], {}\n    if mode == "sentence_transformers_prompt":\n        return texts, {"prompt_name": str(spec["query_prompt_name"])}\n    raise ValueError(f"Unknown query_mode={mode!r}")\n\n\ndef load_encoder(spec: dict[str, Any], *, device: str) -> Any:\n    """Load a SentenceTransformer in inference mode without remote inference APIs."""\n    from sentence_transformers import SentenceTransformer\n\n    model = SentenceTransformer(str(spec["model_id"]), device=device)\n    model.max_seq_length = int(spec["max_length"])\n    if device == "cuda":\n        model.half()\n    return model\n\n\ndef model_revision(model: Any) -> str:\n    """Best-effort resolved Hugging Face commit, without relying on a cache path."""\n    candidates = [model]\n    try:\n        candidates.append(model._first_module())\n    except (AttributeError, TypeError):\n        pass\n    for candidate in list(candidates):\n        auto_model = getattr(candidate, "auto_model", None)\n        if auto_model is not None:\n            candidates.append(auto_model)\n    for candidate in candidates:\n        config = getattr(candidate, "config", None)\n        revision = getattr(config, "_commit_hash", None)\n        if revision:\n            return str(revision)\n    return "unavailable"\n\n\ndef encode_texts(\n    model: Any,\n    spec: dict[str, Any],\n    texts: list[str],\n    *,\n    is_query: bool,\n    device: str,\n    pool: Any | None = None,\n) -> np.ndarray:\n    """Encode texts on one device or a reusable SentenceTransformer GPU pool."""\n    prepared, extra_kwargs = prepare_model_texts(texts, spec, is_query=is_query)\n    batch_size = int(spec["batch_size"] if device == "cuda" else min(8, int(spec["batch_size"])))\n    if pool is not None:\n        # 1,000 texts per inter-process task keeps IPC bounded for 189k items.\n        extra_kwargs.update({"pool": pool, "chunk_size": 1_000})\n    vectors = model.encode(\n        prepared,\n        batch_size=batch_size,\n        show_progress_bar=True,\n        convert_to_numpy=True,\n        normalize_embeddings=True,\n        **extra_kwargs,\n    )\n    return np.asarray(vectors, dtype=np.float32)\n\n\ndef exact_category_search(\n    query_embeddings: np.ndarray,\n    item_embeddings: np.ndarray,\n    query_categories: list[object],\n    category_to_indices: dict[str, np.ndarray],\n    item_ids: np.ndarray,\n    *,\n    top_k: int,\n    query_batch_size: int = 128,\n) -> tuple[list[list[str]], int]:\n    """Exact cosine/IP retrieval inside M0 category partitions.\n\n    Embeddings are L2-normalized.  The function uses a bounded query batch so\n    it does not allocate the full query-by-corpus score matrix.\n    """\n    if query_embeddings.shape[1] != item_embeddings.shape[1]:\n        raise ValueError("Query and item embedding dimensions differ.")\n    rankings: list[list[str]] = [[] for _ in range(len(query_embeddings))]\n    all_indices = np.arange(len(item_ids), dtype=np.int64)\n    fallback_count = 0\n    positions_by_category: dict[str, list[int]] = {}\n    for position, category in enumerate(query_categories):\n        positions_by_category.setdefault(str(category), []).append(position)\n\n    for category, positions in positions_by_category.items():\n        allowed = category_to_indices.get(category)\n        if allowed is None or len(allowed) == 0:\n            allowed = all_indices\n            fallback_count += len(positions)\n        k = min(top_k, len(allowed))\n        if k == 0:\n            continue\n        for start in range(0, len(positions), query_batch_size):\n            batch_positions = positions[start : start + query_batch_size]\n            scores = query_embeddings[batch_positions] @ item_embeddings[allowed].T\n            top_local = np.argpartition(scores, kth=scores.shape[1] - k, axis=1)[:, -k:]\n            top_scores = np.take_along_axis(scores, top_local, axis=1)\n            order = np.argsort(top_scores, axis=1)[:, ::-1]\n            top_global = allowed[np.take_along_axis(top_local, order, axis=1)]\n            for query_position, item_positions in zip(batch_positions, top_global, strict=True):\n                rankings[query_position] = item_ids[item_positions].astype(str).tolist()\n    return rankings, fallback_count\n\n\ndef macro_recall_at_k(rankings: list[list[str]], gold_sets: list[frozenset[str]], k: int) -> float:\n    return float(\n        np.mean(\n            [\n                len(set(prediction[:k]) & relevant) / len(relevant)\n                for prediction, relevant in zip(rankings, gold_sets, strict=True)\n            ]\n        )\n    )\n\n\ndef hit_rate_at_k(rankings: list[list[str]], gold_sets: list[frozenset[str]], k: int) -> float:\n    return float(\n        np.mean(\n            [\n                bool(set(prediction[:k]) & relevant)\n                for prediction, relevant in zip(rankings, gold_sets, strict=True)\n            ]\n        )\n    )\n\n\ndef evaluate_rankings(\n    rankings: list[list[str]],\n    gold_sets: list[frozenset[str]],\n    *,\n    model_name: str,\n    model_id: str,\n    item_encoding_seconds: float,\n    query_encoding_seconds: float,\n    search_seconds: float,\n    embedding_dimension: int,\n    category_fallback_queries: int,\n) -> dict[str, Any]:\n    record: dict[str, Any] = {\n        "model": model_name,\n        "model_id": model_id,\n        "embedding_dimension": int(embedding_dimension),\n        "item_encoding_seconds": float(item_encoding_seconds),\n        "query_encoding_seconds": float(query_encoding_seconds),\n        "search_seconds": float(search_seconds),\n        "category_fallback_queries": int(category_fallback_queries),\n        "mean_candidates": float(np.mean([len(row) for row in rankings])),\n    }\n    for k in METRIC_KS:\n        record[f"recall@{k}"] = macro_recall_at_k(rankings, gold_sets, k)\n    record["hit_rate@50"] = hit_rate_at_k(rankings, gold_sets, 50)\n    return record\n\n\ndef save_rankings_jsonl(path: Path, query_groups: pd.Series, rankings: list[list[str]]) -> None:\n    """Persist validation candidates for later union/error analysis, not an index."""\n    with path.open("w", encoding="utf-8") as handle:\n        for query_group, item_ids in zip(query_groups, rankings, strict=True):\n            handle.write(\n                json.dumps(\n                    {"query_group": int(query_group), "candidate_item_ids": item_ids},\n                    ensure_ascii=False,\n                )\n                + "\\n"\n            )\n\n\ndef hardware_snapshot() -> dict[str, Any]:\n    """Return safe hardware facts for ClearML and the reproducibility manifest."""\n    try:\n        import torch\n\n        if torch.cuda.is_available():\n            properties = torch.cuda.get_device_properties(0)\n            return {\n                "device": "cuda",\n                "gpu_name": properties.name,\n                "gpu_total_memory_gb": round(properties.total_memory / 1024**3, 2),\n            }\n    except ImportError:\n        pass\n    return {"device": "cpu"}\n', shared_module.__dict__)
sys.modules['dense_retrieval'] = shared_module

from dense_retrieval import (
    METRIC_KS, MODEL_SPECS, compose_item_text, compose_query_text,
    encode_texts, evaluate_rankings, exact_category_search, hardware_snapshot,
    load_encoder, load_frozen_proxy, model_revision, save_rankings_jsonl,
)

In [ ]:
unknown_models = set(MODEL_NAMES) - set(MODEL_SPECS)
assert not unknown_models, f"Unknown models: {sorted(unknown_models)}"
load_started = time.perf_counter()
proxy = load_frozen_proxy(TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH, seed=SEED)
validation_queries = proxy["validation_queries"]
gold_sets = proxy["gold_sets"]
candidate_items = proxy["candidate_items"]
item_texts = compose_item_text(candidate_items, description_char_limit=DESCRIPTION_CHAR_LIMIT)
validation_query_texts = compose_query_text(validation_queries)
hardware = hardware_snapshot()
device = str(hardware["device"])
assert device == "cuda", "Enable a Kaggle GPU accelerator before running M3."
import torch

visible_gpu_devices = [f"cuda:{index}" for index in range(torch.cuda.device_count())]
encoding_devices = visible_gpu_devices if USE_ALL_VISIBLE_GPUS else visible_gpu_devices[:1]
assert encoding_devices, "No CUDA device is visible to PyTorch."
multi_gpu_enabled = len(encoding_devices) > 1
clearml_task.set_parameter("runtime/encoding_devices", list(encoding_devices))

display(pd.DataFrame([
    {"split": "proxy_train", "positive_rows": len(proxy["proxy_train_pairs"]), "query_groups": proxy["proxy_train_pairs"]["query_group"].nunique()},
    {"split": "validation", "positive_rows": len(proxy["proxy_valid_pairs"]), "query_groups": len(validation_queries)},
]))
display(pd.DataFrame([
    {"name": name, "model_id": MODEL_SPECS[name]["model_id"], "query_mode": MODEL_SPECS[name]["query_mode"], "max_length": MODEL_SPECS[name]["max_length"], "gpu_batch_size": MODEL_SPECS[name]["batch_size"]}
    for name in MODEL_NAMES
]))
print({
    "load_seconds": round(time.perf_counter() - load_started, 2),
    "hardware": hardware,
    "encoding_devices": encoding_devices,
    "multi_gpu_enabled": multi_gpu_enabled,
    "category_oracle_recall": proxy["category_oracle"],
})

## 4. E05a — exact direct dense retrieval

Запускается только `multilingual_e5_large_instruct`. При наличии двух GPU кодирование корпуса
распределяется между ними, а exact category search остаётся прежним. Это
изолированный запуск: его результат можно сравнивать с другими моделями по
одинаковому frozen split и Recall@50.

In [ ]:
import torch

def run_direct_dense_model(model_name: str) -> tuple[dict[str, object], list[list[str]]]:
    spec = MODEL_SPECS[model_name]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = load_encoder(spec, device="cpu" if multi_gpu_enabled else device)
    pool = None
    item_embeddings = query_embeddings = None
    try:
        resolved_revision = model_revision(model)
        if multi_gpu_enabled:
            # Sentence Transformers distributes independent batches over both GPUs.
            pool = model.start_multi_process_pool(target_devices=encoding_devices)

        item_started = time.perf_counter()
        item_embeddings = encode_texts(
            model, spec, item_texts, is_query=False, device=device, pool=pool,
        )
        item_seconds = time.perf_counter() - item_started
        query_started = time.perf_counter()
        query_embeddings = encode_texts(
            model, spec, validation_query_texts, is_query=True, device=device, pool=pool,
        )
        query_seconds = time.perf_counter() - query_started
        search_started = time.perf_counter()
        rankings, fallback_queries = exact_category_search(
            query_embeddings,
            item_embeddings,
            validation_queries["search_category"].tolist(),
            proxy["category_to_indices"],
            proxy["candidate_item_ids"],
            top_k=TOP_K,
            query_batch_size=QUERY_SCORE_BATCH_SIZE,
        )
        search_seconds = time.perf_counter() - search_started
        record = evaluate_rankings(
            rankings,
            gold_sets,
            model_name=model_name,
            model_id=str(spec["model_id"]),
            item_encoding_seconds=item_seconds,
            query_encoding_seconds=query_seconds,
            search_seconds=search_seconds,
            embedding_dimension=item_embeddings.shape[1],
            category_fallback_queries=fallback_queries,
        )
        record.update({
            "status": "completed",
            "wall_seconds": time.perf_counter() - started,
            # Worker-process peak VRAM is not available in the parent process.
            "peak_vram_mb": float("nan") if multi_gpu_enabled else float(torch.cuda.max_memory_allocated() / 1024**2),
            "max_length": int(spec["max_length"]),
            "query_mode": str(spec["query_mode"]),
            "model_revision": resolved_revision,
            "encoding_devices": ",".join(encoding_devices),
        })
        return record, rankings
    finally:
        if pool is not None:
            model.stop_multi_process_pool(pool)
        del model, item_embeddings, query_embeddings
        gc.collect()
        torch.cuda.empty_cache()


records: list[dict[str, object]] = []
failed_models: dict[str, str] = {}
for model_name in MODEL_NAMES:
    try:
        record, rankings = run_direct_dense_model(model_name)
        ranking_path = OUTPUT_DIR / f"validation_top{TOP_K}__{model_name}.jsonl"
        save_rankings_jsonl(ranking_path, validation_queries["query_group"], rankings)
        record["validation_rankings_path"] = str(ranking_path)
        records.append(record)
        for k in METRIC_KS:
            clearml_logger.report_scalar(f"Recall@{k}", model_name, float(record[f"recall@{k}"]), 0)
        clearml_logger.report_scalar("HitRate@50", model_name, float(record["hit_rate@50"]), 0)
        clearml_logger.report_scalar("Item encoding seconds", model_name, float(record["item_encoding_seconds"]), 0)
        clearml_logger.report_scalar("Search seconds", model_name, float(record["search_seconds"]), 0)
        clearml_logger.report_scalar("Peak VRAM MB", model_name, float(record["peak_vram_mb"]), 0)
        print({"model": model_name, "recall@50": round(float(record["recall@50"]), 6), "wall_seconds": round(float(record["wall_seconds"]), 2)})
    except Exception as exc:
        failed_models[model_name] = f"{type(exc).__name__}: {exc}"
        clearml_logger.report_text(f"M3 model failed: {model_name}: {failed_models[model_name]}")
        torch.cuda.empty_cache()
        print({"model": model_name, "status": "failed", "error": failed_models[model_name]})

# Do not abort here: the results cell persists a failure manifest for this single model.

## 5. Результаты, Kaggle Output и ClearML

In [ ]:
results_frame = pd.DataFrame(records)
if not results_frame.empty:
    results_frame = results_frame.sort_values(
        ["recall@50", "recall@200", "wall_seconds"], ascending=[False, False, True]
    ).reset_index(drop=True)

results_path = OUTPUT_DIR / "m3_e05a_results.csv"
manifest_path = OUTPUT_DIR / "m3_e05a_manifest.json"
results_frame.to_csv(results_path, index=False)
manifest = {
    "stage": "M3_E05a_zero_shot_direct_dense",
    "environment": "kaggle",
    "validation_protocol": "benchmark_aligned_proxy_v1",
    "seed": SEED,
    "top_k": TOP_K,
    "model_requested": MODEL_NAME,
    "models_completed": results_frame["model"].tolist() if not results_frame.empty else [],
    "failed_models": failed_models,
    "hardware": hardware,
    "shared_source_sha256": 'd7d8dcca6d65f25d909730f8f1fcef1245fe9253d25a80e53024a666e1a089f7',
    "source_files": {
        path.name: {"bytes": path.stat().st_size, "modified_ns": path.stat().st_mtime_ns}
        for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH)
    },
    "packages": {
        package: importlib.metadata.version(package)
        for package in ("clearml", "sentence-transformers", "torch", "transformers", "numpy", "pandas")
    },
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

clearml_status = "not reported"
try:
    if not results_frame.empty:
        clearml_logger.report_table("M3 E05a summary", "direct_dense", 0, table_plot=results_frame)
        clearml_task.set_parameter("results/recall_at_50", float(results_frame.iloc[0]["recall@50"]))
    else:
        clearml_logger.report_text(f"M3 model failed: {MODEL_NAME}: {failed_models.get(MODEL_NAME)}")
    clearml_task.upload_artifact("m3_e05a_results", artifact_object=results_path)
    clearml_task.upload_artifact("m3_e05a_manifest", artifact_object=manifest_path)
    clearml_task.set_parameter("results/notebook_seconds", float(time.perf_counter() - NOTEBOOK_STARTED))
    clearml_task.close()
    clearml_status = "metrics and compact artifacts uploaded"
except Exception as exc:
    clearml_status = f"logging failed after local results were saved: {type(exc).__name__}: {exc}"

display(results_frame)
print({
    "model": MODEL_NAME,
    "status": "completed" if not results_frame.empty else "failed",
    "recall@50": None if results_frame.empty else round(float(results_frame.iloc[0]["recall@50"]), 6),
    "results_path": str(results_path),
    "manifest_path": str(manifest_path),
    "clearml": clearml_status,
    "notebook_seconds": round(time.perf_counter() - NOTEBOOK_STARTED, 2),
})

## Что сохранить после Run All

Последняя ячейка создаёт `m3_dense__multilingual_e5_large_instruct.zip` в `/kaggle/working`.
Внутри будут `m3_e05a_results.csv`, `m3_e05a_manifest.json` и top-200
validation candidates для `multilingual_e5_large_instruct`. Затем создайте Kaggle Version и
скачайте ZIP. Не выгружайте model cache и временные embeddings. После
завершения всех пяти изолированных запусков сравним модели по Recall@50 и
выберем не более двух для E05b/hybrid.

## 6. ZIP-архив результатов

Финальная ячейка проверяет ключевые артефакты и создаёт небольшой ZIP для скачивания. В него не попадают model cache и временные embeddings.

In [ ]:
# Архивируем только итоговые артефакты эксперимента для скачивания.
from pathlib import Path
from shutil import make_archive
from IPython.display import FileLink, display

output_dir = Path(OUTPUT_DIR)
assert output_dir.exists(), f"Не найдена папка с результатами: {output_dir}"

expected_files = ["m3_e05a_results.csv", "m3_e05a_manifest.json"]
missing = [name for name in expected_files if not (output_dir / name).exists()]
assert not missing, f"Не созданы ожидаемые файлы: {missing}"

if not results_frame.empty:
    ranking_files = sorted(output_dir.glob("validation_top*.jsonl"))
    assert ranking_files, "Не сохранён top-200 validation candidates файл."

zip_path = Path("/kaggle/working/m3_dense__multilingual_e5_large_instruct.zip")
make_archive(
    base_name=str(zip_path.with_suffix("")),
    format="zip",
    root_dir=str(output_dir.parent),
    base_dir=output_dir.name,
)

print(f"ZIP создан: {zip_path}")
print("Файлы в архивируемой папке:", sorted(path.name for path in output_dir.iterdir()))
display(FileLink(zip_path))